# Cost Evolution

Per-iteration extraction cost trajectories produced by `scripts/measure_cost.py`.
Each cost function gets its own subplot; per-term lines are normalized by their
iteration-0 cost and colored by initial cost (viridis, log-scaled). A bold black
median is overlaid.

**Note:** trajectories are padded with last-observation-carried-forward (LOCF),
i.e. once a term stops, its final cost is held constant out to `max_iters`. This
treats termination as "converged and stopped improving" and keeps the per-iteration
sample population fixed, at the cost of overstating stability near the tail.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors

plt.rcParams.update({"figure.dpi": 120})

In [ ]:
# Configuration: point at a seed-terms run directory.
RUN_DIR = Path("../data/seed_terms/posh-notepad")

with (RUN_DIR / "cost_evolution.json").open() as f:
    results: dict[str, list[dict[str, int]]] = json.load(f)

trajectories = [traj for traj in results.values() if traj]
cost_fns = sorted({k for traj in trajectories for k in traj[0].keys()})
max_iters = max(len(traj) for traj in trajectories)
print(f"{len(results)} terms loaded, {len(trajectories)} with data")
print(f"cost functions: {cost_fns}")
print(f"max iterations: {max_iters}")

In [ ]:
def cost_matrix(trajectories, cost_fn, max_iters):
    """Return (locf_normalized_rows, initial_costs) for one cost function.

    Rows are normalized by the iteration-0 cost so every row starts at 1.0,
    then padded out to `max_iters` with last-observation-carried-forward:
    once a term stops, its final normalized cost is held constant. This
    keeps the per-iteration sample population fixed across the x-axis.
    """
    rows, initials = [], []
    for traj in trajectories:
        vals = [d[cost_fn] for d in traj if cost_fn in d]
        if not vals or vals[0] == 0:
            continue
        row = np.full(max_iters, np.nan)
        for i, v in enumerate(vals[:max_iters]):
            row[i] = v
        # LOCF: hold the last observed value out to max_iters
        row[len(vals) :] = vals[-1]
        initials.append(vals[0])
        rows.append(row / vals[0])
    return np.vstack(rows), np.array(initials)


n = len(cost_fns)
ncols = 2 if n > 1 else 1
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)
cmap = plt.get_cmap("viridis")

for ax, cost_fn in zip(axes.flat, cost_fns):
    arr, initials = cost_matrix(trajectories, cost_fn, max_iters)
    if len(arr) == 0:
        ax.set_title(f"{cost_fn} (no data)")
        continue

    norm = colors.LogNorm(vmin=max(1.0, initials.min()), vmax=initials.max())
    x = np.arange(max_iters)
    for row, init in zip(arr, initials):
        ax.plot(x, row, color=cmap(norm(init)), alpha=0.15, lw=0.8)

    median = np.median(arr, axis=0)
    ax.plot(x, median, color="black", lw=2.0, label="median")

    ax.set_title(cost_fn)
    ax.set_xlabel("iteration")
    ax.set_ylabel("cost / initial cost (LOCF)")
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best", fontsize=8)

    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label="initial cost")

for ax in axes.flat[n:]:
    ax.set_visible(False)

fig.suptitle(f"cost evolution ({len(trajectories)} terms, {RUN_DIR.name}, LOCF padded)")
fig.tight_layout()
plt.show()